# Pipeline Sequencial — Modelo 7 → Modelo 5

## Task 1 — Contrato de integração

Este notebook implementará futuramente o pipeline sequencial formado por:

1. **Modelo 7 — CatBoost:** previsão de risco por sistema PSD, no nível de estação e plataforma;
2. **Modelo 5 — Random Forest:** previsão de risco por porta individual dentro das plataformas selecionadas.

Nesta primeira task ainda não há execução dos modelos. O objetivo é registrar o contrato que deverá ser respeitado nas próximas etapas.

## 1. Fluxo acordado

```text
Dados disponíveis até o final do dia D
                    ↓
Modelo 7 — CatBoost por plataforma
                    ↓
Plataformas com probabilidade >= 0,317380
                    ↓
Modelo 5 — Random Forest por porta
                    ↓
Portas priorizadas para inspeção
```

O Modelo 7 é executado primeiro por trabalhar no nível mais amplo e indicar que alguma porta da plataforma poderá falhar. O Modelo 5 recebe somente as portas pertencentes às plataformas alertadas e tenta localizar qual porta deve ser priorizada. A diferença de granularidade é proposital: o CatBoost ajuda o Random Forest reduzindo o espaço de busca. O score do CatBoost não será usado como feature do Random Forest nesta primeira versão, porque o Modelo 5 não foi treinado com essa variável.

## 2. Comparação das interfaces atuais

| Item | Modelo 7 — CatBoost | Modelo 5 — Random Forest | Situação |
|---|---|---|---|
| Granularidade | Plataforma × dia | Porta × dia | Compatível, complementar e proposital |
| Momento da previsão | Final de `D`, incluindo eventos de `D` | Início de `D`, usando eventos até `D-1` | Precisa alinhar |
| Horizonte | `D+1` a `D+7` | `(D, D+7]` | Mesmo tamanho; referência precisa alinhar |
| Falha positiva | ZM01 com `falha_flag == True` | Qualquer ZM01 | Precisa corrigir |
| Plataforma | `id_psd = BUT_P1` | `id_plat = BUT_1` | Precisa converter |
| Porta | Não é a unidade prevista | `id_porta = BUT_1_05` | Precisa padronizar |
| Cobertura | 11 estações e 22 plataformas | 10 estações e 480 portas | `FRE` sem detalhamento por porta |
| Features | 21 registradas em metadata | 71 mantidas por SHAP | Congelar lista do Modelo 5 |
| Limiar | `0,317380` | `0,5183` | Manter separados |
| Artefato | `.cbm` e metadata disponíveis | Modelo não serializado | Precisa exportar |

## 3. Decisões do contrato

### Data e target comuns

- O pipeline será iniciado **ao final do dia `D`**.
- O Modelo 7 manterá as features disponíveis até o final de `D`.
- O Modelo 5 manterá inicialmente sua regra original, usando o histórico até `D-1`.
- Os dois modelos observarão o horizonte futuro de `D+1` a `D+7`.
- O Modelo 7 manterá o target de falha confirmada; o Modelo 5 manterá inicialmente o target de qualquer ZM01 da porta.
- O alinhamento completo de tempo e target será um aprimoramento posterior, não um bloqueio do primeiro pipeline.

### Chaves canônicas

```text
estacao = BUT
plataforma = 1
id_psd = BUT_P1
id_porta = BUT_P1_PDM05
```

As chaves internas dos modelos serão convertidas para esse padrão na fronteira do pipeline. Uma porta só poderá chegar ao Modelo 5 quando pertencer à mesma estação e plataforma alertadas pelo Modelo 7.

### Cobertura

Enquanto o Modelo 5 não incluir `FRE`, um alerta do Modelo 7 nessa estação deverá permanecer como alerta de plataforma sem detalhamento por porta. O alerta não poderá ser descartado silenciosamente.

## 4. Resultado da Task 1

Os modelos são conceitualmente complementares e a diferença de granularidade é o mecanismo central do pipeline: o Modelo 7 encontra a plataforma em risco e o Modelo 5 procura a porta dentro dela. Para automatizar a primeira versão, será necessário:

1. salvar a lista ordenada das 71 features;
2. serializar o Random Forest e registrar seu limiar e metadata;
3. implementar a conversão das chaves;
4. tratar plataformas sem cobertura no Modelo 5;
5. preservar a regra temporal original de cada modelo durante a inferência.

Em uma iteração posterior, o Modelo 5 poderá ser retreinado com o mesmo instante e o mesmo target do Modelo 7 para permitir uma avaliação unificada de ponta a ponta. Isso não impede o primeiro teste do encadeamento.

A especificação completa está em `docs/src/content/docs/sprints/Sprint 3/pipeline-sequencial-modelos.mdx`.

# Task 2 — Preparar o Modelo 5 para inferência

Esta etapa reconstrói a configuração selecionada do Random Forest sem modificar os arquivos das sprints anteriores. O painel e as ordens antigas são somente entradas de leitura. Como o objeto treinado na Sprint 2 não foi serializado, o modelo é retreinado e seu limiar é recalculado somente na validação.

Os novos artefatos serão gravados exclusivamente em `models/pipeline-sequencial/artefatos/`:

- `modelo_5_random_forest.joblib`;
- `metadata_modelo_5.json`.

In [1]:
%pip install pyarrow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


def localizar_raiz_repositorio():
    caminho_atual = Path.cwd().resolve()
    for candidato in (caminho_atual, *caminho_atual.parents):
        if (candidato / 'models/modelo-5/painel.parquet').exists():
            return candidato
    raise FileNotFoundError('Não foi possível localizar a raiz do repositório g04.')


RAIZ_REPOSITORIO = localizar_raiz_repositorio()
DIRETORIO_MODELO_5 = RAIZ_REPOSITORIO / 'models/modelo-5'
DIRETORIO_ARTEFATOS = RAIZ_REPOSITORIO / 'models/pipeline-sequencial/artefatos'
DIRETORIO_ARTEFATOS.mkdir(parents=True, exist_ok=True)

caminho_painel = DIRETORIO_MODELO_5 / 'painel.parquet'
caminho_zm01 = RAIZ_REPOSITORIO / 'data/Corretiva Emergencial/processados/ZM01_limpa.xlsx'

painel = pd.read_parquet(caminho_painel)
print(f'Painel carregado: {painel.shape[0]:,} linhas × {painel.shape[1]} colunas')
print(f'Período: {painel["Data"].min().date()} a {painel["Data"].max().date()}')
print(f'Portas: {painel["id_porta"].nunique()}')

Painel carregado: 306,864 linhas × 83 colunas
Período: 2023-11-01 a 2025-10-31
Portas: 480


In [3]:
# Reproduz somente as features criadas dentro do notebook original do Modelo 5.
# Nenhum arquivo antigo é alterado.
PAT_LOC = re.compile(
    r'^SP04-6-PSD-([A-Z]{3})(?:-PLT(\d)|-P(\d)S\d)?(?:-PP(\d{2})(PDM|PFX|PSE))?'
)


def carregar_zm01_porta(caminho):
    ordens = pd.read_excel(caminho, sheet_name='ZM01_limpa')
    correspondencias = ordens['local_de_instalacao'].apply(
        lambda valor: PAT_LOC.match(valor) if isinstance(valor, str) else None
    )
    ordens['est'] = [item.group(1) if item else None for item in correspondencias]
    ordens['via'] = [
        (item.group(2) or item.group(3)) if item else None
        for item in correspondencias
    ]
    ordens['porta'] = [
        int(item.group(4)) if item and item.group(4) else None
        for item in correspondencias
    ]
    ordens = ordens.dropna(
        subset=['est', 'via', 'porta', 'inicio_avaria_datetime']
    ).copy()
    ordens['porta'] = ordens['porta'].astype(int)
    ordens['id_porta'] = (
        ordens['est'] + '_' + ordens['via'] + '_'
        + ordens['porta'].astype(str).str.zfill(2)
    )
    ordens['Data'] = pd.to_datetime(ordens['inicio_avaria_datetime']).dt.normalize()
    return ordens


zm01 = carregar_zm01_porta(caminho_zm01)
eventos = (
    zm01[['id_porta', 'Data']]
    .dropna()
    .drop_duplicates()
    .sort_values(['id_porta', 'Data'])
    .reset_index(drop=True)
)
eventos['ordem'] = eventos.groupby('id_porta').cumcount() + 1
eventos['diff'] = eventos.groupby('id_porta')['Data'].diff().dt.days
eventos['intervalo_medio_ate_aqui'] = (
    eventos.groupby('id_porta')['diff']
    .apply(lambda serie: serie.expanding().mean())
    .reset_index(level=0, drop=True)
)

referencias = painel[['id_porta', 'Data']].copy()
referencias['_idx'] = painel.index
historico = pd.merge_asof(
    referencias.sort_values('Data'),
    eventos[['id_porta', 'Data', 'ordem', 'intervalo_medio_ate_aqui']].sort_values('Data'),
    on='Data',
    by='id_porta',
    direction='backward',
    allow_exact_matches=False,
)

painel['n_corretivas_historico_total'] = (
    pd.Series(historico['ordem'].values, index=historico['_idx'].values)
    .reindex(painel.index)
    .fillna(0)
)
intervalos = pd.Series(
    historico['intervalo_medio_ate_aqui'].values,
    index=historico['_idx'].values,
).reindex(painel.index)
painel['intervalo_medio_zm01'] = intervalos.fillna(intervalos.median())
painel['razao_atraso_zm01'] = painel['dias_desde_zm01'] / painel['intervalo_medio_zm01']
painel['dias_desde_ultima_corretiva'] = painel[['dias_desde_zm01', 'dias_desde_zm02']].min(axis=1)

painel['rampa_7d'] = painel['n_total_7d'] - (painel['n_total_14d'] - painel['n_total_7d'])
painel['razao_7d_30d'] = painel['n_total_7d'] / (painel['n_total_30d'] / 4 + 1)
painel['frac_tecnicos_30d'] = painel['n_tecnicos_30d'] / (painel['n_total_30d'] + 1)
painel['est_cod'] = painel['est'].astype('category').cat.codes
painel['via_cod'] = painel['via'].astype(int)

print('Features adicionais do notebook original reproduzidas.')

Features adicionais do notebook original reproduzidas.


In [4]:
COLUNAS_EXCLUIDAS = {
    'id_porta', 'Data', 'est', 'via', 'id_plat', 'y',
    'n_falhas_7d_futuro', 'cobertura_30d', 'mes',
}
FEATURES_REMOVIDAS_SHAP = [
    'n_CSD_3d', 'n_FO_3d', 'zm02_30d', 'n_RCE_7d',
    'n_FO_1d', 'n_OA_1d', 'dia_semana', 'n_RCE_3d',
    'n_CSD_1d', 'n_CA_1d', 'fim_de_semana', 'n_RCE_1d',
]

features_candidatas = [
    coluna for coluna in painel.columns if coluna not in COLUNAS_EXCLUIDAS
]
features_modelo_5 = [
    coluna for coluna in features_candidatas
    if coluna not in FEATURES_REMOVIDAS_SHAP
]

assert len(features_candidatas) == 83, (
    f'Esperadas 83 candidatas, encontradas {len(features_candidatas)}.'
)
assert len(features_modelo_5) == 71, (
    f'Esperadas 71 features finais, encontradas {len(features_modelo_5)}.'
)
assert not painel[features_modelo_5].isna().any().any()
assert np.isfinite(painel[features_modelo_5].to_numpy()).all()

print(f'Features candidatas: {len(features_candidatas)}')
print(f'Features removidas: {len(FEATURES_REMOVIDAS_SHAP)}')
print(f'Features finais congeladas: {len(features_modelo_5)}')
print(features_modelo_5)

Features candidatas: 83
Features removidas: 12
Features finais congeladas: 71
['num_porta', 'n_FC_1d', 'n_LSD_1d', 'n_OBS_1d', 'n_total_1d', 'n_long30_1d', 'n_tecnicos_1d', 'n_CA_3d', 'n_FC_3d', 'n_LSD_3d', 'n_OA_3d', 'n_OBS_3d', 'n_total_3d', 'n_long30_3d', 'n_tecnicos_3d', 'n_CA_7d', 'n_CSD_7d', 'n_FC_7d', 'n_FO_7d', 'n_LSD_7d', 'n_OA_7d', 'n_OBS_7d', 'n_total_7d', 'n_long30_7d', 'n_tecnicos_7d', 'n_CA_14d', 'n_CSD_14d', 'n_FC_14d', 'n_FO_14d', 'n_LSD_14d', 'n_OA_14d', 'n_OBS_14d', 'n_RCE_14d', 'n_total_14d', 'n_long30_14d', 'n_tecnicos_14d', 'n_CA_30d', 'n_CSD_30d', 'n_FC_30d', 'n_FO_30d', 'n_LSD_30d', 'n_OA_30d', 'n_OBS_30d', 'n_RCE_30d', 'n_total_30d', 'n_long30_30d', 'n_tecnicos_30d', 'dur_max_7d', 'dur_max_30d', 'n_total_decaido_h3', 'n_total_decaido_h7', 'n_total_decaido_h14', 'n_tecnicos_decaido_h7', 'n_LSD_decaido_h7', 'n_mdu_7d', 'dias_desde_zm01', 'zm01_30d', 'zm01_90d', 'dias_desde_zm02', 'zm02_90d', 'dias_desde_zm03', 'zm03_90d', 'n_corretivas_historico_total', 'intervalo

In [5]:
FIM_TREINO = pd.Timestamp('2025-04-23')
INICIO_TESTE = pd.Timestamp('2025-05-01')
INICIO_VALIDACAO = pd.Timestamp('2025-01-31')

treino = painel[painel['Data'] <= FIM_TREINO].copy()
teste = painel[painel['Data'] >= INICIO_TESTE].copy()
treino_fit = treino[
    treino['Data'] <= INICIO_VALIDACAO - pd.Timedelta(days=8)
].copy()
validacao = treino[treino['Data'] >= INICIO_VALIDACAO].copy()


def selecionar_limiar(y_real, probabilidades):
    candidatos = np.unique(
        np.quantile(probabilidades, np.linspace(0.50, 0.999, 200))
    )
    return max(
        (
            f1_score(
                y_real,
                (probabilidades >= limiar).astype(int),
                zero_division=0,
            ),
            limiar,
        )
        for limiar in candidatos
    )[1]


parametros_modelo_5 = {
    'n_estimators': 400,
    'max_depth': 14,
    'min_samples_leaf': 40,
    'class_weight': 'balanced',
    'n_jobs': -1,
    'random_state': 42,
}

modelo_validacao = RandomForestClassifier(**parametros_modelo_5)
modelo_validacao.fit(
    treino_fit[features_modelo_5],
    treino_fit['y'],
)
probabilidades_validacao = modelo_validacao.predict_proba(
    validacao[features_modelo_5]
)[:, 1]
limiar_modelo_5 = selecionar_limiar(
    validacao['y'], probabilidades_validacao
)

modelo_5 = RandomForestClassifier(**parametros_modelo_5)
modelo_5.fit(treino[features_modelo_5], treino['y'])
probabilidades_teste = modelo_5.predict_proba(teste[features_modelo_5])[:, 1]
predicoes_teste = (probabilidades_teste >= limiar_modelo_5).astype(int)
tn, fp, fn, tp = confusion_matrix(teste['y'], predicoes_teste).ravel()
metricas_teste = {
    'precision': float(precision_score(teste['y'], predicoes_teste, zero_division=0)),
    'recall': float(recall_score(teste['y'], predicoes_teste, zero_division=0)),
    'f1': float(f1_score(teste['y'], predicoes_teste, zero_division=0)),
    'pr_auc': float(average_precision_score(teste['y'], probabilidades_teste)),
    'roc_auc': float(roc_auc_score(teste['y'], probabilidades_teste)),
    'alertas': int(predicoes_teste.sum()),
    'tn': int(tn),
    'fp': int(fp),
    'fn': int(fn),
    'tp': int(tp),
}

print(f'Limiar selecionado: {limiar_modelo_5:.6f}')
print(json.dumps(metricas_teste, indent=2, ensure_ascii=False))

Limiar selecionado: 0.488753
{
  "precision": 0.1451476171169511,
  "recall": 0.63495646565624,
  "f1": 0.23628236282362824,
  "pr_auc": 0.18533521575736303,
  "roc_auc": 0.6967904891797934,
  "alertas": 27131,
  "tn": 41789,
  "fp": 23193,
  "fn": 2264,
  "tp": 3938
}


In [6]:
caminho_modelo_5 = DIRETORIO_ARTEFATOS / 'modelo_5_random_forest.joblib'
caminho_metadata_5 = DIRETORIO_ARTEFATOS / 'metadata_modelo_5.json'

metadata_modelo_5 = {
    'modelo': 'Modelo 5 — Random Forest',
    'algoritmo': 'RandomForestClassifier',
    'arquivo_modelo': caminho_modelo_5.name,
    'fonte_original': 'models/modelo-5/modelo_5.ipynb',
    'unidade_analise': 'porta_dia',
    'chave_interna': 'id_porta no formato EST_VIA_PORTA',
    'momento_previsao': 'inicio do dia D',
    'dados_permitidos': 'eventos anteriores ao dia D',
    'horizonte': '(D, D+7]',
    'target': 'y',
    'definicao_target': 'existe qualquer ZM01 da mesma porta em (D, D+7]',
    'limiar': float(limiar_modelo_5),
    'quantidade_features': len(features_modelo_5),
    'features': features_modelo_5,
    'tipos_features': {
        coluna: str(painel[coluna].dtype)
        for coluna in features_modelo_5
    },
    'criterio_features': (
        'mesmo conjunto de 71 features mantidas após a seleção SHAP; '
        'ordem determinística herdada do painel'
    ),
    'features_removidas_shap': FEATURES_REMOVIDAS_SHAP,
    'hiperparametros': parametros_modelo_5,
    'periodos': {
        'treino_inicio': str(treino['Data'].min().date()),
        'treino_fim': str(treino['Data'].max().date()),
        'validacao_inicio': str(validacao['Data'].min().date()),
        'validacao_fim': str(validacao['Data'].max().date()),
        'teste_inicio': str(teste['Data'].min().date()),
        'teste_fim': str(teste['Data'].max().date()),
    },
    'metricas_teste': metricas_teste,
    'bibliotecas': {
        'python': __import__('sys').version.split()[0],
        'pandas': pd.__version__,
        'numpy': np.__version__,
        'scikit_learn': sklearn.__version__,
        'joblib': joblib.__version__,
    },
    'gerado_em_utc': datetime.now(timezone.utc).isoformat(),
}

joblib.dump(modelo_5, caminho_modelo_5, compress=3)
caminho_metadata_5.write_text(
    json.dumps(metadata_modelo_5, indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8',
)

modelo_recarregado = joblib.load(caminho_modelo_5)
assert list(modelo_recarregado.feature_names_in_) == features_modelo_5
probabilidades_recarregadas = modelo_recarregado.predict_proba(
    teste[features_modelo_5]
)[:, 1]
assert np.allclose(probabilidades_recarregadas, probabilidades_teste)

print(f'Modelo salvo e validado em: {caminho_modelo_5}')
print(f'Metadata salvo em: {caminho_metadata_5}')

Modelo salvo e validado em: /home/kaian/Documentos/modulo7/g04/models/pipeline-sequencial/artefatos/modelo_5_random_forest.joblib
Metadata salvo em: /home/kaian/Documentos/modulo7/g04/models/pipeline-sequencial/artefatos/metadata_modelo_5.json
